# Laboratório: MCP

**Dupla:**

- Nome 1:
- Nome 2:

Neste laboratório deverão ser construídos um servidor MCP e uma skill para um domínio escolhido pela dupla. O servidor é usado por um agente que também se conecta a pelo menos um servidor MCP da comunidade, e a skill ensina esse agente a fazer uma tarefa do domínio com as ferramentas do servidor. O domínio pode ser o mesmo do laboratório de RAG ou do agente feito antes, e nesse caso o retriever ou as ferramentas construídas lá viram ferramentas do servidor, ou pode ser um domínio novo. Cada seção descreve o que se espera dela e traz um espaço para o código, e podem ser criadas quantas células forem precisas.

Um servidor MCP pode ser usado por qualquer cliente, com modelos e mensagens de sistema que a dupla não controla. Tudo o que um agente sabe sobre o servidor vem dos nomes, das docstrings e dos parâmetros, então essas descrições devem ser escritas para quem nunca viu o código.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph "langchain[mcp]" uv

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
import os
import sys
from pathlib import Path

os.environ["FASTMCP_LOG_LEVEL"] = "WARNING"

import pandas as pd
import yaml

from fastmcp import Client
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.mcp import MCPAdapter
from langchain.messages import HumanMessage
from langchain.tools import tool

## Servidor

O servidor deve ser escrito com o `FastMCP` em um arquivo `server.py`, criado pelo grupo na mesma pasta do notebook e entregue junto com ele. Ele deve oferecer:

- **Ferramentas:** o retriever do RAG ou as ferramentas do agente anterior, se o domínio for o mesmo, e pelo menos duas outras ferramentas, que podem consultar uma API pública, calcular algo ou ler dados próprios do domínio.
- **Recurso:** pelo menos um, com URI fixa ou template, com dados que a aplicação anexa ao contexto.
- **Prompt:** pelo menos um, com argumentos.

O arquivo segue este formato:

```python
from fastmcp import FastMCP

mcp = FastMCP("nome-do-servidor")

# Ferramentas, recursos e prompts aqui.

if __name__ == "__main__":
    mcp.run(show_banner=False)
```

O servidor usa o transporte stdio: o cliente inicia o `server.py` como um processo separado a cada conexão e conversa com ele pela entrada e saída padrão, sem porta nem URL. Por ser outro processo, o servidor não enxerga as variáveis definidas aqui, então tudo de que ele precisa, como o modelo de embedding e o caminho do índice, deve ser criado dentro do arquivo.

### Índice do RAG

Se o servidor usa o retriever, o índice do vector store precisa existir na pasta em que o servidor roda. No Colab, a pasta do índice pode ser enviada compactada e descompactada aqui, ou o índice pode ser refeito com o código de indexação do laboratório de RAG. O modelo de embedding do servidor deve ser o mesmo usado para criar o índice.

### Teste do servidor

O servidor deve ser testado com um cliente que liste as ferramentas, os recursos e os prompts e que chame cada ferramenta pelo menos uma vez, incluindo um caso com argumento inválido. O `MCPAdapter` converte só as ferramentas; os recursos e os prompts são listados e lidos com o `Client` do FastMCP. Os dois recebem o caminho do `server.py` e iniciam o servidor sozinhos, e um erro no arquivo aparece como exceção na célula que abre a conexão.

In [ ]:
SERVER = Path("server.py")

async with MCPAdapter(SERVER) as adapter:
    own_tools = await adapter.list_tools()

pd.DataFrame([{"name": tool.name, "description": tool.description, "args": list(tool.args)} for tool in own_tools])

## Cliente

O cliente é um agente conectado ao servidor do grupo e a pelo menos um servidor MCP da comunidade, remoto ou local, cada um adicionado ao dicionário `mcpServers`. Com mais de um servidor, o `MCPAdapter` prefixa o nome de cada ferramenta com a chave do servidor, o que evita colisão quando dois servidores têm ferramentas com o mesmo nome. Cada ferramenta abre a conexão quando é chamada, e chamar o agente dentro de `async with adapter` faz as chamadas de uma pergunta compartilharem uma conexão.

In [ ]:
servers = {
    "nosso": {"command": sys.executable, "args": ["server.py"]},
    # "time": {"command": "uvx", "args": ["mcp-server-time"]},
}

adapter = MCPAdapter({"mcpServers": servers})

### Skill

A skill é uma pasta em `skills/` com um `SKILL.md`, e a pasta e o arquivo devem ser criados pela dupla: o frontmatter traz `name` e `description`, e o corpo traz as instruções. Ela deve ensinar uma tarefa do domínio que precise de vários passos, e não repetir o que uma ferramenta já faz sozinha:

- **Descrição:** diz o que a skill faz e quando usá-la, com as palavras que um usuário usaria no pedido.
- **Corpo:** indica quais ferramentas chamar e em que ordem, usando pelo menos duas do servidor da dupla, o que fazer quando uma falhar ou não encontrar nada e o formato exato da resposta. A skill pode combinar também ferramentas do servidor da comunidade. As ferramentas são citadas pelo nome que o agente vê, com a chave do servidor como prefixo, como `nosso_search_recipes`.

A ferramenta `load_skill` lê o `SKILL.md` pelo nome da skill, e o catálogo, que vai na mensagem de sistema, inclui o nome e a descrição de todas as skills da pasta `skills/`.

In [ ]:
@tool
def load_skill(name: str) -> str:
    """Carrega as instruções completas de uma skill pelo nome."""
    return Path("skills", name, "SKILL.md").read_text()


metadata = [yaml.safe_load(path.read_text().split("---")[1]) for path in sorted(Path("skills").glob("*/SKILL.md"))]
catalog = "\n".join(f"- {meta['name']}: {meta['description']}" for meta in metadata)
print(catalog)

### Catálogo

Tabela com o servidor, o nome, a descrição e os argumentos de cada ferramenta disponível, seguida de uma avaliação das descrições do servidor da comunidade: quais deixam claro quando a ferramenta deve ser usada e quais são ambíguas ou se sobrepõem às do servidor da dupla.

### Agente

Agente criado com `create_agent`, com as ferramentas de todos os servidores mais `load_skill` e com o catálogo de skills na mensagem de sistema. Devem ser feitas pelo menos sete perguntas, mostrando para cada uma as ferramentas chamadas e a resposta:

- pelo menos duas que dependam do servidor da dupla;
- pelo menos uma que dependa do servidor da comunidade;
- uma que precise combinar ferramentas dos dois servidores;
- uma que deva carregar a skill e uma parecida que não deva;
- uma que nenhum servidor responda.

In [ ]:
async with adapter:
    tools = await adapter.list_tools()

# system_prompt = ""

# agent = create_agent(model, tools=tools + [load_skill], system_prompt=system_prompt)